# Задание №4. Разработка спецификации требований к информационной системе (SRS)

## Веб-сервис семантической сегментации аэрофотоснимков

### Введение

Спецификация требований к программному обеспечению (SRS) является одним из основных документов в управлении проектами создания информационных систем. Этот документ служит связующим звеном между заказчиком и разработчиками, определяя чёткое понимание того, что должна делать система, каким требованиям она должна соответствовать и как будет оцениваться успешность проекта.

Согласно статистике Standish Group, около 70 % IT-проектов терпят неудачу из-за неправильно определённых или непонятых требований.

Грамотно составленная спецификация требований позволяет:

- Снизить риски проекта за счёт раннего выявления противоречий и неясностей
- Обеспечить общее понимание целей между всеми заинтересованными сторонами
- Создать основу для планирования, разработки и тестирования системы
- Предотвратить «расползание границ проекта»
- Сформировать критерии приёмки системы

## 1. ВВЕДЕНИЕ

### 1.1. Цель документа

Настоящий документ содержит совокупность требований к **веб-сервису семантической сегментации аэрофотоснимков и спутниковых снимков** (далее — Система), разрабатываемому в рамках дипломного проекта. Документ служит основой для проектирования, реализации и приёмочного тестирования Системы.

### 1.2. Область применения

Система предназначена для автоматической семантической сегментации растровых аэрофотоснимков и снимков дистанционного зондирования Земли. Пользователь загружает изображение, Система обрабатывает его нейросетевой моделью и возвращает результаты — цветовую карту сегментации и количественные показатели покрытия по классам объектов. Регистрация учётных записей и долгосрочное хранение данных в рамках системы не предусмотрены.

### 1.3. Определения и сокращения

| Термин | Определение |
|--------|-------------|
| ДЗЗ | Дистанционное зондирование Земли |
| Семантическая сегментация | Задача попиксельной классификации изображения с отнесением каждого пикселя к одному из заранее определённых классов объектов |
| Маска сегментации | Растровое изображение, в котором значение каждого пикселя соответствует идентификатору семантического класса |
| mIoU | Средний показатель пересечения по объединению (Mean Intersection over Union) — основная метрика качества семантической сегментации |
| mDice | Средний коэффициент Дайса по классам |
| Общая пиксельная точность | Доля пикселей, классифицированных верно, от общего числа пикселей изображения |
| Программный интерфейс | Набор правил и форматов, по которым внешние программы могут обращаться к сервису |
| Сеансовый идентификатор | Временный идентификатор, связывающий загруженное изображение с результатами его обработки в рамках одного сеанса работы |
| Нейросетевая модель | Обученная нейронная сеть архитектуры HyenaPixel, выполняющая сегментацию изображения |

### 1.4. Пользователи системы

Система рассчитана на единственную категорию пользователей — **специалиста в области дистанционного зондирования или геоинформатики** — с умеренным уровнем технической грамотности. Пользователь умеет работать с веб-браузером, понимает понятие «семантические классы объектов» и способен интерпретировать числовые показатели качества сегментации. Никакой установки программного обеспечения от пользователя не требуется.

## 2. ФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 2.1. Загрузка изображения

**ФТ-001: Загрузка файла изображения**

- Описание: пользователь должен иметь возможность загрузить растровый снимок для последующей обработки
- Входные данные: файл в формате JPEG, PNG или TIFF; необязательное название снимка (произвольная текстовая строка)
- Выходные данные: сеансовый идентификатор загруженного изображения; уведомление о постановке задачи в очередь на обработку
- Ограничения: максимальный размер файла — 50 МБ; минимальное разрешение — 256×256 пикселей; максимальное разрешение — 4096×4096 пикселей
- Приоритет: **Критический**
- Критерий проверки: файл успешно принимается сервером; при нарушении ограничений пользователь получает информативное сообщение об ошибке с указанием причины

---

**ФТ-002: Предварительный просмотр загруженного изображения**

- Описание: после загрузки файла система должна отображать уменьшенную копию (миниатюру) исходного изображения, чтобы пользователь мог убедиться в правильности выбранного файла
- Выходные данные: миниатюра размером не более 512×512 пикселей
- Приоритет: **Высокий**
- Критерий проверки: миниатюра отображается в течение 2 секунд после завершения загрузки файла

### 2.2. Обработка и сегментация

**ФТ-003: Автоматическая семантическая сегментация**

- Описание: система должна автоматически запускать сегментацию загруженного изображения нейросетевой моделью без каких-либо дополнительных действий со стороны пользователя
- Входные данные: загруженное изображение, привязанное к сеансовому идентификатору
- Выходные данные: маска сегментации в формате PNG с кодированием класса в значении пикселя; статус задачи, обновляемый в реальном времени
- Семантические классы: здание, дорожное покрытие, водная поверхность, растительность, открытая почва, транспортные средства, прочее (неопределённые объекты)
- Бизнес-правила: задача обрабатывается асинхронно; статус задачи обновляется каждые 3 секунды без перезагрузки страницы; при превышении времени ожидания в очереди более 5 минут пользователь получает соответствующее уведомление
- Приоритет: **Критический**
- Критерий проверки: сегментация изображения разрешением 1024×1024 пикселя завершается за время не более 20 секунд; выходная маска не содержит пикселей с идентификаторами, выходящими за пределы установленного перечня классов

---

**ФТ-004: Отображение прогресса обработки**

- Описание: в период ожидания результатов система должна информировать пользователя о текущем состоянии задачи
- Выходные данные: индикатор выполнения с текстовым описанием текущего этапа («Загрузка завершена», «Идёт обработка», «Готово», «Ошибка обработки»)
- Приоритет: **Высокий**
- Критерий проверки: состояние задачи корректно отображается на каждом этапе; при ошибке обработки пользователь получает понятное описание проблемы

### 2.3. Отображение результатов

**ФТ-005: Интерактивная визуализация маски сегментации**

- Описание: после завершения сегментации система должна отображать результат в виде цветовой маски, наложенной на исходное изображение, с возможностью управления отображением
- Функциональность:
  - наложение цветовой маски с регулируемой прозрачностью (ползунок от 0 до 100 %)
  - масштабирование колесом мыши и перемещение зажатой кнопкой мыши
  - отображение наименования семантического класса при наведении курсора
  - переключение между режимами «только исходное», «только маска», «наложение»
  - включение и отключение отдельных классов в визуализации
- Приоритет: **Критический**
- Критерий проверки: визуализатор корректно отображает результаты для изображений всех допустимых разрешений; задержка при переключении режимов не превышает 300 мс

---

**ФТ-006: Показатели качества сегментации**

- Описание: система должна рассчитывать и отображать количественные показатели результата сегментации
- Выходные данные:
  - сводная таблица метрик: общая пиксельная точность, средняя точность по классам, mIoU, mDice
  - таблица поклассных значений: доля площади (%), IoU, коэффициент Дайса для каждого из семи классов
  - круговая диаграмма распределения пикселей по классам
- Приоритет: **Высокий**
- Критерий проверки: числовые значения метрик совпадают с результатами независимого расчёта на тех же данных с точностью до четвёртого знака после запятой

---

**ФТ-007: Цветовая легенда классов**

- Описание: рядом с картой сегментации должна постоянно отображаться легенда, связывающая цвет маски с наименованием семантического класса
- Выходные данные: блок легенды с цветовым образцом, наименованием каждого класса и его долей площади в процентах от общей площади снимка
- Приоритет: **Высокий**
- Критерий проверки: легенда обновляется при включении и отключении классов; процентные значения соответствуют данным таблицы из ФТ-006

### 2.4. Выгрузка результатов

**ФТ-008: Сохранение результатов на устройство пользователя**

- Описание: пользователь должен иметь возможность сохранить результаты обработки на своё устройство
- Варианты выгрузки:
  - маска сегментации в формате PNG
  - изображение с наложенной маской в формате PNG (в текущих настройках прозрачности)
  - таблица метрик в формате CSV
- Приоритет: **Средний**
- Критерий проверки: скачанные файлы открываются корректно; данные в CSV соответствуют значениям, отображённым в интерфейсе

---

**ФТ-009: Повторная обработка без повторной загрузки**

- Описание: пользователь должен иметь возможность запустить повторную обработку уже загруженного изображения, не загружая файл заново
- Функциональность: кнопка «Обработать заново» на странице результатов; в рамках текущего сеанса исходный файл остаётся доступным на сервере до закрытия браузерной вкладки
- Приоритет: **Низкий**
- Критерий проверки: повторная обработка запускается корректно; старые результаты заменяются новыми

## 3. НЕФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 3.1. Требования к производительности

**НФТ-П-001: Время отклика интерфейса**

- Переходы между страницами, переключение режимов визуализации и фильтрация данных в таблицах должны выполняться за время не более 1 секунды
- Загрузка файла размером до 50 МБ должна начинаться немедленно, а индикатор прогресса обновляться в реальном времени

**НФТ-П-002: Производительность обработки**

- Время сегментации изображения разрешением 512×512 пикселей — не более 5 секунд
- Время сегментации изображения разрешением 1024×1024 пикселя — не более 20 секунд
- Время сегментации изображения разрешением 2048×2048 пикселей — не более 60 секунд

**НФТ-П-003: Начальная загрузка страницы**

- Главная страница сервиса должна полностью загружаться за время не более 3 секунд при скорости интернет-соединения от 10 Мбит/с

### 3.2. Требования к надёжности

**НФТ-Н-001: Обработка ошибок при сегментации**

- При сбое нейросетевой обработки система должна автоматически повторить попытку не более двух раз; после двух неудачных попыток задаче присваивается статус «Ошибка», пользователю отображается описание проблемы и предложение повторить загрузку
- Сбой при обработке одного изображения не должен нарушать работу сервиса для других пользователей

**НФТ-Н-002: Устойчивость к некорректным входным данным**

- Система должна корректно обрабатывать попытки загрузки файлов неподдерживаемых форматов, повреждённых файлов и файлов, превышающих допустимые ограничения, не прерывая работу сервиса
- При каждой подобной попытке пользователю отображается сообщение с указанием причины отказа и допустимых параметров

**НФТ-Н-003: Временное хранение данных сеанса**

- Исходное изображение и результаты сегментации хранятся на сервере в течение 1 часа с момента загрузки; по истечении этого времени они автоматически удаляются
- Пользователь информируется о временном характере хранения данных непосредственно на странице результатов

### 3.3. Требования к удобству использования

**НФТ-Ю-001: Простота освоения**

- Пользователь, впервые открывший сервис, должен самостоятельно выполнить полный цикл работы (загрузка снимка → просмотр карты сегментации → сохранение результатов) за время не более 5 минут без обращения к документации
- Обоснование: целевая аудитория — специалисты со средним уровнем технической грамотности; избыточная сложность интерфейса снижает практическую ценность инструмента

**НФТ-Ю-002: Информативность интерфейса**

- Все активные элементы управления должны иметь визуальную обратную связь при взаимодействии (изменение курсора, подсветка, изменение состояния)
- Все сообщения об ошибках должны содержать конкретное описание проблемы и рекомендацию по её устранению

**НФТ-Ю-003: Адаптивность интерфейса**

- Интерфейс должен корректно отображаться на экранах с разрешением от 1280×768 до 3840×2160 пикселей без горизонтальной прокрутки

**НФТ-Ю-004: Язык интерфейса**

- Все надписи, подсказки, сообщения и наименования классов в интерфейсе должны быть представлены на русском языке

### 3.4. Требования к совместимости

**НФТ-С-001: Браузерная совместимость**

- Система должна корректно работать в актуальных версиях браузеров: Google Chrome, Mozilla Firefox, Microsoft Edge, Safari
- Использование сторонних расширений или плагинов для работы с сервисом не требуется

**НФТ-С-002: Поддерживаемые форматы файлов**

- Входные форматы: JPEG, PNG, TIFF (включая многоканальные GeoTIFF)
- Выходные форматы: PNG (маски и изображения с наложением), CSV (таблицы метрик)

**НФТ-С-003: Программный интерфейс**

- Система должна предоставлять программный интерфейс, позволяющий внешним приложениям отправлять изображения на обработку и получать результаты без использования веб-браузера
- Обмен данными осуществляется в формате JSON; передача файлов — через составной запрос (multipart)
- Документация программного интерфейса должна соответствовать стандарту OpenAPI 3.0

### 3.5. Требования к масштабируемости

**НФТ-М-001: Одновременная обработка нескольких запросов**

- Система должна корректно обрабатывать не менее 5 одновременно поступивших задач сегментации, выстраивая их в очередь без потери данных и без аварийного завершения

**НФТ-М-002: Возможность обновления модели**

- Архитектура серверной части должна обеспечивать замену весов нейросетевой модели без полного перезапуска веб-сервиса; новые запросы после обновления обрабатываются актуальной версией модели

## 4. ТРЕБОВАНИЯ К ИНТЕРФЕЙСАМ

### 4.1. Пользовательский интерфейс

**ТИ-ПИ-001: Главная страница (загрузка снимка)**

- Область перетаскивания файла с чётким визуальным обозначением допустимых форматов и ограничений по размеру
- Альтернативная кнопка выбора файла через стандартный диалог операционной системы
- Краткая инструкция по работе с сервисом (не более 5 шагов)
- Индикатор прогресса загрузки с числовым показателем (процент и объём переданных данных)

**ТИ-ПИ-002: Страница обработки**

- Миниатюра загруженного изображения
- Индикатор текущего состояния задачи с текстовым описанием этапа
- Возможность отменить задачу и вернуться к загрузке нового изображения

**ТИ-ПИ-003: Страница результатов**

- Основная область: интерактивный просмотрщик с исходным изображением и наложенной маской сегментации
- Панель управления слоями: ползунок прозрачности, переключатели режимов просмотра, список классов с флажками включения/отключения
- Боковая панель: цветовая легенда классов с долями площади и числовые значения метрик качества
- Блок кнопок сохранения результатов (маска, изображение с наложением, таблица метрик)

### 4.2. Программный интерфейс

**ТИ-АПИ-001: Пример передачи изображения на обработку**

```
POST /api/v1/segment
Content-Type: multipart/form-data

Тело запроса:
  image: <файл в формате JPEG / PNG / TIFF>
  name:  <необязательное название снимка, строка>

Ответ (202 — запрос принят):
{
  "session_id": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
  "status":     "queued",
  "created_at": "2026-03-24T15:30:00Z"
}
```

**ТИ-АПИ-002: Пример получения результатов**

```
GET /api/v1/segment/{session_id}

Ответ при успешном завершении (200):
{
  "session_id": "...",
  "status": "completed",
  "metrics": {
    "pixel_accuracy": 0.923,
    "mean_class_accuracy": 0.887,
    "mean_iou": 0.771,
    "mean_dice": 0.861,
    "per_class": {
      "building":    { "area_pct": 18.4, "iou": 0.852, "dice": 0.920 },
      "road":        { "area_pct": 12.1, "iou": 0.801, "dice": 0.889 },
      "water":       { "area_pct":  5.3, "iou": 0.934, "dice": 0.966 },
      "vegetation":  { "area_pct": 38.7, "iou": 0.891, "dice": 0.942 },
      "bare_ground": { "area_pct": 22.6, "iou": 0.743, "dice": 0.852 },
      "vehicle":     { "area_pct":  1.4, "iou": 0.612, "dice": 0.759 },
      "other":       { "area_pct":  1.5, "iou": 0.504, "dice": 0.670 }
    }
  },
  "mask_url":    "/api/v1/segment/{session_id}/mask.png",
  "overlay_url": "/api/v1/segment/{session_id}/overlay.png",
  "model_version": "hyenapixel-v1.0",
  "processing_time_s": 11.2
}
```

## 5. ОГРАНИЧЕНИЯ И ДОПУЩЕНИЯ

### 5.1. Ограничения

**ОГР-001: Вычислительная инфраструктура**

Достижение значений производительности, указанных в НФТ-П-002, возможно только при наличии на сервере графического процессора с объёмом видеопамяти не менее 8 ГБ; при работе исключительно на центральном процессоре время обработки возрастает в 5–10 раз.

**ОГР-002: Отсутствие долгосрочного хранения данных**

Система не ведёт архив результатов: после истечения времени жизни сеанса (1 час) все связанные с ним файлы удаляются; ответственность за сохранение результатов лежит на пользователе.

**ОГР-003: Точность модели**

Качество сегментации определяется составом обучающей выборки; значительный визуальный сдвиг (изображения с характеристиками, существенно отличающимися от обучающих данных) может приводить к снижению точности без предупреждения системы.

### 5.2. Допущения

**ДОП-001:** Предполагается, что обучающая выборка содержит не менее 1 000 размеченных изображений ДЗЗ, охватывающих все семь поддерживаемых семантических классов.

**ДОП-002:** Предполагается, что пользователи работают на рабочих станциях с широкополосным интернет-соединением (не менее 10 Мбит/с), достаточным для своевременной загрузки файлов размером до 50 МБ.

**ДОП-003:** Предполагается, что система развёртывается на сервере, доступном по протоколу HTTPS; настройка сетевой инфраструктуры выходит за рамки данной спецификации.

## 6. КРИТЕРИИ ПРИЁМКИ

### 6.1. Функциональная приёмка

**КП-Ф-001:** Все функциональные требования с приоритетом «Критический» (ФТ-001, ФТ-003, ФТ-005) реализованы и успешно протестированы на не менее чем 10 изображениях различного содержания и разрешения.

**КП-Ф-002:** Полный цикл работы «загрузка → сегментация → просмотр → сохранение» выполняется без ошибок в 100 % тестовых случаев.

**КП-Ф-003:** Сохранённые файлы результатов открываются корректно в стандартных программах (графические редакторы, табличный процессор).

### 6.2. Нефункциональная приёмка

**КП-НФ-001:** Значение показателя mIoU на тестовой выборке изображений ДЗЗ составляет не менее 0,65.

**КП-НФ-002:** Сегментация изображения разрешением 1024×1024 пикселя выполняется за время не более 20 секунд на целевой аппаратной платформе.

**КП-НФ-003:** Интерфейс корректно отображается в Chrome, Firefox и Edge актуальных версий при разрешении экрана 1920×1080 и 2560×1440 пикселей.

### 6.3. Тестирование

**КП-Т-001:** Покрытие автоматическими тестами модулей серверной части составляет не менее 70 % по числу строк кода.

**КП-Т-002:** Проведено пользовательское приёмочное тестирование с участием не менее 2 специалистов; оба выполнили полный цикл работы без посторонней помощи за время не более 5 минут.

**КП-Т-003:** Проведено тестирование устойчивости к ошибочному вводу: загрузка файлов неподдерживаемых форматов, повреждённых файлов и файлов с превышением допустимого размера обрабатывается корректно во всех случаях.

**КП-Т-004:** Результаты тестирования нейросетевой модели зафиксированы в отдельном протоколе с указанием поклассных значений IoU, коэффициента Дайса и общей пиксельной точности на тестовой выборке.